# Image Processing/Vision Course Project

## Dataset Exploration

This notebook loads the selected dataset, examines its structure, and visualizes sample images together with their ground-truth annotations.

In [ ]:
!pip install -q datasets pillow matplotlib numpy pandas opencv-python

In [ ]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from datasets import load_dataset

In [ ]:
SEED = 10

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to {SEED}")

In [ ]:
# Repository-relative project paths
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Supports running either from the repository root or from the notebooks folder
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Figures directory:", FIGURES_DIR)


In [ ]:
from torchvision.datasets import VOCSegmentation

dataset = VOCSegmentation(
    root=DATA_DIR,
    year="2012",
    image_set="train",
    download=True
)

print(dataset)
print(f"Number of samples: {len(dataset)}")
print("Each sample contains: (image, segmentation_mask)")

In [ ]:
# Load one sample image and its corresponding ground-truth segmentation mask
image, mask = dataset[0]

print("Image type:", type(image))
print("Mask type:", type(mask))
print("Image size:", image.size)
print("Mask size:", mask.size)

In [ ]:
# Display one sample image next to its ground-truth segmentation mask
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(mask)
axes[1].set_title("Ground-Truth Segmentation Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Inspect the class IDs that appear in this segmentation mask
mask_array = np.array(mask)

unique_class_ids = np.unique(mask_array)

print("Unique class IDs in the mask:")
print(unique_class_ids)
print(f"Number of unique values: {len(unique_class_ids)}")

In [ ]:
# Select a fixed subset of dataset indices for reproducible experiments
NUM_SAMPLES = 100

rng = np.random.default_rng(SEED)

selected_indices = rng.choice(
    len(dataset),
    size=NUM_SAMPLES,
    replace=False
)

selected_indices = sorted(selected_indices.tolist())

print(f"Selected {len(selected_indices)} samples")
print("First 10 selected indices:", selected_indices[:10])

In [ ]:
# Display a few images from the fixed experimental subset (to verify that the sample is diverse)
num_examples = 6
example_indices = selected_indices[:num_examples]

fig, axes = plt.subplots(num_examples, 2, figsize=(12, 4 * num_examples))

for row, index in enumerate(example_indices):
    sample_image, sample_mask = dataset[index]

    axes[row, 0].imshow(sample_image)
    axes[row, 0].set_title(f"Original Image - Dataset Index {index}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(sample_mask)
    axes[row, 1].set_title("Ground-Truth Segmentation Mask")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Count how often each segmentation class appears in the selected sample
class_pixel_counts = {}

for index in selected_indices:
    _, sample_mask = dataset[index]
    sample_mask_array = np.array(sample_mask)

    unique_ids, counts = np.unique(sample_mask_array, return_counts=True)

    for class_id, count in zip(unique_ids, counts):
        class_pixel_counts[class_id] = class_pixel_counts.get(class_id, 0) + int(count)

print("Class IDs found in the selected sample:")
print(sorted(class_pixel_counts.keys()))

In [ ]:
# Map class IDs to readable PASCAL VOC class names
VOC_CLASS_NAMES = {
    0: "background",
    1: "aeroplane",
    2: "bicycle",
    3: "bird",
    4: "boat",
    5: "bottle",
    6: "bus",
    7: "car",
    8: "cat",
    9: "chair",
    10: "cow",
    11: "dining table",
    12: "dog",
    13: "horse",
    14: "motorbike",
    15: "person",
    16: "potted plant",
    17: "sheep",
    18: "sofa",
    19: "train",
    20: "tv/monitor",
    255: "ignore"
}

class_summary = pd.DataFrame([
    {
        "class_id": int(class_id),
        "class_name": VOC_CLASS_NAMES[int(class_id)],
        "pixel_count": pixel_count
    }
    for class_id, pixel_count in class_pixel_counts.items()
])

class_summary = class_summary.sort_values("class_id").reset_index(drop=True)

display(class_summary)

In [ ]:
# Save the selected dataset indices and class distribution for reproducibility

selected_indices_df = pd.DataFrame({
    "dataset_index": selected_indices
})

selected_indices_path = RESULTS_DIR / "selected_indices.csv"
class_summary_path = RESULTS_DIR / "class_summary.csv"

selected_indices_df.to_csv(selected_indices_path, index=False)
class_summary.to_csv(class_summary_path, index=False)

print("Saved:", selected_indices_path)
print("Saved:", class_summary_path)


In [ ]:
# Inspect image dimensions in the selected sample
image_sizes = []

for index in selected_indices:
    sample_image, _ = dataset[index]
    width, height = sample_image.size

    image_sizes.append({
        "dataset_index": index,
        "width": width,
        "height": height
    })

image_sizes_df = pd.DataFrame(image_sizes)

print("Minimum width:", image_sizes_df["width"].min())
print("Maximum width:", image_sizes_df["width"].max())
print("Minimum height:", image_sizes_df["height"].min())
print("Maximum height:", image_sizes_df["height"].max())